# Double Pendulum

Generalised coordinates :
$$
P(\theta_1,\theta_2)
$$
Cartesian coordinates (pendulum 1) :
$$
\begin{cases}
x_1=l_1\sin\theta_1\\
y_1=-l_1\cos\theta_1
\end{cases}
$$
Cartesian coordinates (pendulum 2) :
$$
\begin{cases}
x_2=l_1\sin\theta_1+l_2\sin\theta_2\\
y_2=-(l_1\cos\theta_1+l_2\cos\theta_2)
\end{cases}
$$
---

Newton's Laws :
$$
\begin{align}
(m_1+m_2)l_1\ddot{\theta}_1+m_2l_2\ddot{\theta}_2\cos(\theta_2-\theta_1)&=m_2l_2\dot{\theta}_2^2\sin(\theta_2-\theta_1)-(m_1+m_2)g\sin\theta_1\\
l_2\ddot{\theta}_1+l_1\ddot{\theta}_2\cos(\theta_2-\theta_1)&=-l_1\dot{\theta}_1^2\sin(\theta_2-\theta_1)-g\sin\theta_2
\end{align}
$$
In matrix form :
$$
A\,\vec{\ddot{\theta}}=\vec{b}
$$
$$
\text{where}\quad
\begin{cases}
\begin{align*}
\vec{\ddot{\theta}}&=\begin{bmatrix}
\ddot{\theta}_1\\
\ddot{\theta}_2
\end{bmatrix},\\\\
A &= \begin{bmatrix}
(m_1+m_2)l_1 & m_2l_2\cos(\theta_2-\theta_1) \\
l_2 & l_1\cos(\theta_2-\theta_1)
\end{bmatrix},\\\\
\vec{b} &= \begin{bmatrix}
m_2l_2\dot{\theta}_2^2\sin(\theta_2-\theta_1)-(m_1+m_2)g\sin\theta_1\\
-l_1\dot{\theta}_1^2\sin(\theta_2-\theta_1)-g\sin\theta_2
\end{bmatrix}.
\end{align*}
\end{cases}
$$
---

Converting into system of first order ODEs :

$$
\text{Let}\quad \omega_1=\frac{d\theta_1}{dt}\quad\text{and}\quad \omega_2=\frac{d\theta_2}{dt}
$$

Out state vector and its derivitive is :
$$
y=\begin{pmatrix}
\theta_1 \\
\theta_2 \\
\omega_1 \\
\omega_2
\end{pmatrix}\quad\text{and}\quad
\frac{dy}{dt}=\begin{pmatrix}
\omega_1 \\
\omega_2 \\
\alpha_1 \\
\alpha_2
\end{pmatrix},\quad\text{where}\quad \alpha_1=\frac{d\omega_1}{dt}=\ddot{\theta}_1,\quad\alpha_2=\frac{d\omega_2}{dt}=\ddot{\theta}_2
$$

---

In [203]:
import numpy as np
from scipy.linalg import solve

def double_pendulum_derivatives(t, y, m1, m2, l1, l2, g):
    """
    Returns the derivatives of the double pendulum system.
    """
    # Parameters
    θ1, θ2, ω1, ω2 = y # state vector
    delta = θ2 - θ1
    m12 = m1 + m2
    
    # Defining the vectors as matrix functions
    A = np.array([
        [m12 * l1, m2 * l2 * np.cos(delta)],
        [l2, l1 * np.cos(delta)]
    ])
    
    b = np.array([
        m2 * l2 * abs(ω2**2) * np.sin(delta) - m12 * g * np.sin(θ1),
        -l1 * abs(ω1**2) * np.sin(delta) - g * np.sin(θ2)
    ])
    
    # Solve for acceleration
    α1, α2 = solve(A, b)
    
    return np.array([ω1, ω2, α1, α2])

def rk4_step(f, t, y, dt, *args):
    """
    4th Order Runge-Kutta step
    """
    k1 = f(t, y, *args)
    k2 = f(t + dt/2, y + dt/2 * k1, *args)
    k3 = f(t + dt/2, y + dt/2 * k2, *args)
    k4 = f(t + dt, y + dt * k3, *args)

    return y + dt/6 * (k1 + 2*k2 + 2*k3 + k4)

In [192]:
def simulate_double_pendulum(θ1, θ2, ω1=0, ω2=0, 
                             t_max=10, dt=0.01, 
                             m1=1, m2=1, 
                             l1=1, l2=1, 
                             g=9.81):
    """
    Simulate the double pendulum
    """
    # Initial state
    y = np.array([θ1, θ2, ω1, ω2])
    
    # Time array
    t_values = np.arange(0, t_max, dt)
    
    # Store results
    results = np.zeros((len(t_values), 4))
    results[0] = y
    
    # Simulation loop
    for i in range(1, len(t_values)):
        # print(i)
        results[i] = rk4_step(double_pendulum_derivatives, t_values[i-1], 
                             results[i-1], dt, m1, m2, l1, l2, g)

    
    return t_values, results

In [202]:
import matplotlib.pyplot as plt

# SIMPLE TEST CASE
if __name__ == "__main__":
    print("Simple Double Pendulum Test")
    print("Initial conditions: θ1=π/4, θ2=π/2, both starting from rest")
    
    # Run simulation
    t, results = simulate_double_pendulum(
        θ1=90*np.pi/180,    # 45 degrees
        θ2=45*np.pi/180,    # 90 degrees  
        ω1=0,          # starting from rest
        ω2=0,          # starting from rest
        t_max=10,            # simulate for 10 seconds
        dt=0.01              # time step of 0.01 seconds
    )
    
    # Extract results
    theta1, theta2, omega1, omega2 = results.T
    
    # Print some basic info
    print(f"Simulation completed!")
    print(f"Time steps: {len(t)}")
    print(f"Final angles: θ1={theta1[-1]:.3f} rad, θ2={theta2[-1]:.3f} rad")
    print(f"Final velocities: ω1={omega1[-1]:.3f} rad/s, ω2={omega2[-1]:.3f} rad/s")
    
    # Simple plot
    plt.figure(figsize=(10, 6))
    plt.plot(t, theta1 * 180 / np.pi, label='θ₁ (first pendulum)')
    plt.plot(t, theta2 * 180 / np.pi, label='θ₂ (second pendulum)')
    plt.xlabel('Time (s)')
    plt.ylabel('Angle (rad)')
    plt.title('Double Pendulum Motion')
    plt.legend()
    plt.grid(True)
    plt.show()

Simple Double Pendulum Test
Initial conditions: θ1=π/4, θ2=π/2, both starting from rest


/tmp/ipykernel_52153/1158947790.py:20: RuntimeWarning: overflow encountered in scalar power
  m2 * l2 * abs(ω2**2) * np.sin(delta) - m12 * g * np.sin(θ1),
/tmp/ipykernel_52153/1158947790.py:21: RuntimeWarning: overflow encountered in scalar power
  -l1 * abs(ω1**2) * np.sin(delta) - g * np.sin(θ2)
/tmp/ipykernel_52153/1158947790.py:41: RuntimeWarning: invalid value encountered in add
  results = y + dt/6 * (k1 + 2*k2 + 2*k3 + k4)
/tmp/ipykernel_52153/1158947790.py:31: RuntimeWarning: invalid value encountered in remainder
  return (angle + np.pi) % (2 * np.pi) - np.pi


ValueError: array must not contain infs or NaNs

## Attempt 2

In [1]:
# 1. Helper functions

# Matrix b
def b1(theta1, theta2, v_theta2):
    return m2 * l2 * abs(v_theta2 ** 2) * np.sin(theta2 - theta1) - (m1 + m2) * g * np.sin(theta1)

def b2(theta1, theta2, v_theta1):
    return -l1 * abs(v_theta1 ** 2) * np.sin(theta2 - theta1) - g * np.sin(theta2)

# Matrix A
def A11():
    return (m1 + m2) * l1

def A12(theta1, theta2):
    return m2 * l2 * np.cos(theta2 - theta1)

def A21():
    return l1 * np.cos(theta2 - theta1)

def A22(theta1, theta2):
    return l2

# Cartesian Pendulum 1
def x_bob1(theta1):
    return l1 * np.sin(theta1)

def y_bob1(theta1):
    return -l1 * np.cos(theta1)

# Cartesian Pendulum 2
def x_bob2(theta1, theta2):
    return l1 * np.sin(theta1) + l2 * np.sin(theta2)

def y_bob2(theta1, theta2):
    return -(l1 * np.cos(theta1) + l2 * np.cos(theta2))

In [8]:
# 2. Calculates arrays of data points
import numpy as np
from scipy.linalg import solve

# Parameters
m1 = 1
m2 = 1
l1 = 1
l2 = 1
g = 9.81

# Initial conditions
theta1 = -65 * np.pi / 180 # 45 degrees
theta2 = 155 * np.pi / 180 # 45 degrees
v_theta1 = 0
v_theta2 = 0
A = np.array([[A11(), A12(theta1, theta2)],
    [A21(), A22(theta1, theta2)]])
b = np.array([b1(theta1, theta2, v_theta2), b2(theta1, theta2, v_theta1)])
a_theta = solve(A, b)

# Arrays to keep x and y values
x1 = [x_bob1(theta1)] # Bob 1
y1 = [y_bob1(theta1)]
x2 = [x_bob2(theta1, theta2)] # Bob 2
y2 = [y_bob2(theta1, theta2)]

# Keep for animation
i_theta1 = theta1
i_theta2 = theta2

# Loop
i = 1
dt = 0.05 # Time divison
sec = 10
total_i = int(sec/dt)

while i < total_i:
    # Update angular velocity
    v_theta1 += a_theta[0] * dt
    v_theta2 += a_theta[1] * dt

    # Update angle
    theta1 += v_theta1 * dt
    theta2 += v_theta2 * dt

    # Append new x and y coordinates
    x1.append(x_bob1(theta1)) # Bob 1
    y1.append(y_bob1(theta1))
    x2.append(x_bob2(theta1, theta2)) # Bob 2
    y2.append(y_bob2(theta1, theta2))

    # Update angular velocity for next iteration
    A = np.array([[A11(), A12(theta1, theta2)],
                  [A21(), A22(theta1, theta2)]])
    b = np.array([b1(theta1, theta2, v_theta2), b2(theta1, theta2, v_theta1)])
    a_theta = solve(A, b)

    # Update iteration count
    i += 1
